## `sparkly_lite` from Python

This notebook performs the work of the [tutorial](doc/tutorial.md) from Python, with the functions `block` and `evaluate`. It repeats Parts 1, 2, 4, 5 and 7 of the tutorial, reads the candidate pairs into Python, where they can be examined or joined to other records, and measures their recall. We recommend going through Parts 1 to 7 of the tutorial first, since this notebook uses the terms they introduce, such as K, the reduction and the recall.

Run the cells in order with Shift + Enter. The text above each cell says what the cell prints. Run times may vary depending on your machine.

### 1. The smoke test

This cell checks that `sparkly_lite` runs on this Mac, with a search on two records, as the installer does. It prints `OK — sparkly_lite runs on this machine.`

In [ ]:
!sparkly-lite smoke

### 2. Blocking two tables and measuring recall

`sl.block` performs the same run as `sparkly-lite block` and prints the same summary. This cell blocks the two tables of `data/dblp_acm` with K = 10 and measures recall against the gold set, as Parts 1 and 2 of the tutorial do with K = 20. The summary shows 22,940 candidates, a reduction of 99.6177%, and 2,223 of the 2,224 gold pairs, a recall of 99.96%.

In [ ]:
import sparkly_lite as sl

A, B, G = ("data/dblp_acm/table_a.parquet",
           "data/dblp_acm/table_b.parquet",
           "data/dblp_acm/gold.parquet")

summary = sl.block(A, B, gold=G, k=10)

The numbers of the summary are also in the `dict` that `sl.block` returns, so that a script can use them. This cell prints four of them, `n_candidates`, `reduction`, `recall` and `seconds`. The first three are 22,940, 99.6177% and 99.96%.

In [ ]:
print("candidates:", f"{summary['n_candidates']:,}")
print("reduction: ", f"{summary['reduction']:.4%}")
print("recall:    ", f"{summary['recall']:.2%}")
print("time:      ", f"{summary['seconds']:.2f} s")

### 3. Saving and reading the candidate pairs

`out` saves the candidate pairs to a file, as `--out` does in Part 3 of the tutorial, and `pyarrow`, which is installed with `sparkly_lite`, reads the file into Python. This cell blocks on the columns `title` and `authors` only, as Part 5 of the tutorial does, with K = 10, saves the pairs to `candidates.parquet`, and prints the first 10.

The file holds 22,874 pairs. The first rows are the candidates of the record of table B whose id is 0, with the best first. Its best candidate, the record of table A whose id is 2123, has a score of 60.777, and the next one 20.978. `id1` holds the ids of table A, `id2` the ids of table B, and a higher score means more similar records.

In [ ]:
import pyarrow.parquet as pq

sl.block(A, B, columns=["title", "authors"], k=10, out="candidates.parquet", verbose=False)

pairs = pq.read_table("candidates.parquet")
print(f"{pairs.num_rows:,} candidate pairs\n")
print("  id1    id2     score")
for row in pairs.slice(0, 10).to_pylist():
    print(f"{row['id1']:>5}  {row['id2']:>5}  {row['blocking_score']:8.3f}")

### 4. Measuring the recall of saved candidates

`sl.evaluate` measures the recall of a candidate set that was saved earlier against a gold set, without blocking again. This cell measures the pairs that the previous cell saved. It prints `n_candidates` 22,874, `n_gold` 2,224, `tp` 2,222 and `fn` 2, which are the gold pairs found and missing, and a recall of 0.9991, or 99.91%.

In [ ]:
report = sl.evaluate("candidates.parquet", G)
print(report)

### 5. Choosing K

A smaller K gives fewer candidates and less work for the matcher, and a larger K keeps more of the true matches. This cell runs `sl.block` at six values of K and prints the candidates, the reduction and the recall of each, as Part 4 of the tutorial does for two values. It prints the following.

```
   K   candidates   reduction    recall
   1        2,294    99.9618%    99.10%
   2        4,588    99.9235%    99.51%
   5       11,470    99.8089%    99.87%
  10       22,940    99.6177%    99.96%
  20       45,880    99.2355%   100.00%
  50      114,700    98.0887%   100.00%
```

On these tables the recall reaches 100% at K = 20, with 45,880 candidates.

In [ ]:
print(f"{'K':>4}  {'candidates':>11}  {'reduction':>10}  {'recall':>8}")
for k in [1, 2, 5, 10, 20, 50]:
    s = sl.block(A, B, gold=G, k=k, verbose=False, progress=False)
    print(f"{k:>4}  {s['n_candidates']:>11,}  {s['reduction']:>10.4%}  {s['recall']:>8.2%}")

### 6. Finding the duplicates in a table of 1,000,000 records

This cell finds the duplicates in the table of 1,000,000 songs in `data/music_1m`, with `dedup=True`, as Part 7 of the tutorial does. The table forms 499,999,500,000 distinct pairs. The cell takes a few minutes, and the progress line shows the number of rows processed and the estimated time remaining. When it finishes, the summary shows 12,530,036 candidates, a reduction of 99.9975%, and 141,925 of the 146,011 gold pairs, a recall of 97.20%.

In [ ]:
summary = sl.block("data/music_1m/table_a.parquet",
                   gold="data/music_1m/gold.parquet",
                   dedup=True, k=20)

### Going further

- The paths of `A` and `B` can be replaced by files in any of the formats that the section [File formats](doc/technical-guide.md#file-formats) of the technical guide lists, or by pandas or Polars DataFrames or pyarrow Tables.
- `sl.block` selects the blocking columns with `columns=[...]`, and the tokenizer with `tokenizer="trigram"`. The [technical guide](doc/technical-guide.md) describes every option, with the Python name of each option of the command.
- Before using your own tables, check [What your files need](doc/technical-guide.md#what-your-files-need), including how to prepare record IDs.